In [5]:
path0 = './'
filename = 'syntax_matched_sentences.txt'
threshold = 0.4  # adjust this value as needed

data = []
with open(f"{path0}{filename}", "r") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 5:
            sentence1, sentence2, tags1, tags2, word_overlap = parts
            try:
                if float(word_overlap) < threshold and tags1 == tags2:
                    data.append((sentence1, sentence2, tags1, word_overlap))
            except ValueError:
                continue  # skip lines with non-numeric word_overlap

print(len(data))
print(data[-1])

# removing entries with tag counts less than 5
from collections import Counter
tag_counts = Counter(tags for _, _, tags, _ in data)
data = [entry for entry in data if tag_counts[entry[2]] > 5]

print(f"Filtered size: {len(data)}")

2369
('The old tradition and a custom changed gradually', 'The soft pillow and a blanket lay comfortably', 'DT JJ NN CC DT NN VBD RB', '0.375')
Filtered size: 2098


In [9]:
from collections import Counter
import os # Import os module to get basename of file paths
import random # Import random module for picking random sentences

# --- Configuration ---
path0 = './'
filename = 'syntax_matched_sentences.txt'
threshold = 0.4

sentences_file0 = "/home/acevedo/syn-sem/datasets/txt/sem/second/matching/english/sentences0.txt"
sentences_file1 = "/home/acevedo/syn-sem/datasets/txt/sem/second/matching/english/sentences1.txt"

# --- Step 1: Load allowed sentences and map them to their line numbers AND source file identifier ---
# The map will now store a tuple: (line_num, file_identifier)
sentence_to_line_map = {}
try:
    # Read from the first file
    with open(sentences_file0, 'r') as f:
        # enumerate(f) defaults to starting from 0
        for line_num, line in enumerate(f):
            # Store a tuple of (line_number, '0' for sentences_file0)
            sentence_to_line_map[line.strip()] = (line_num, '0')
            
    # Read from the second file, overwriting duplicates if any (and updating file identifier)
    with open(sentences_file1, 'r') as f:
        # enumerate(f) defaults to starting from 0
        for line_num, line in enumerate(f):
            # Store a tuple of (line_number, '1' for sentences_file1)
            sentence_to_line_map[line.strip()] = (line_num, '1')
            
    print(f"Loaded {len(sentence_to_line_map)} unique sentences to filter against.")
except FileNotFoundError as e:
    print(f"Error: Sentence file not found - {e}")
    exit() # Exit if the files are missing

# --- Step 2: Initial filtering from the source file ---
data = []
with open(f"{path0}{filename}", "r") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 5:
            sentence1, sentence2, tags1, tags2, word_overlap = parts
            try:
                if float(word_overlap) < threshold and tags1 == tags2:
                    data.append((sentence1, sentence2, tags1, word_overlap))
            except ValueError:
                # Skip lines where word_overlap cannot be converted to float
                continue

print(f"Size after initial overlap/tag filter: {len(data)}")

# --- Step 3: Filter by tag frequency ---
# Filters out entries where the associated tag appears 5 or fewer times in the dataset.
tag_counts = Counter(tags for _, _, tags, _ in data)
data = [entry for entry in data if tag_counts[entry[2]] > 5]
print(f"Size after tag count filter: {len(data)}")

# --- Step 4: Filter by sentence existence and ensure unique labels and matching data length ---
# This step now ensures that both final_data and semantic_labels contain unique entries
# (based on the label and file identifier) and that their lengths match.
final_data = []
# Use a set to track which (label, file_identifier) pairs have already been processed
processed_label_identifiers = set() 
semantic_labels_to_export = [] # This will store (label, file_identifier) tuples for export
# New dictionary to map (label, file_identifier) to the original (sentence1, sentence2) from syntax file
label_identifier_to_syntax_sentences_map = {}

allowed_sentences_keys = sentence_to_line_map.keys()

for entry in data: # 'data' here is the result of Step 3
    sentence1, sentence2, _, _ = entry
    
    current_label_identifier = None
    if sentence1 in allowed_sentences_keys:
        current_label_identifier = sentence_to_line_map[sentence1]
    elif sentence2 in allowed_sentences_keys:
        current_label_identifier = sentence_to_line_map[sentence2]

    # Only add the entry to final_data and its label to semantic_labels_to_export
    # if a valid label was found AND this (label, file_identifier) hasn't been processed yet.
    if current_label_identifier and current_label_identifier not in processed_label_identifiers:
        final_data.append(entry)
        semantic_labels_to_export.append(current_label_identifier)
        processed_label_identifiers.add(current_label_identifier)
        # Store the original sentences from the syntax file for verification
        label_identifier_to_syntax_sentences_map[current_label_identifier] = (sentence1, sentence2)

# Update the main data list with the final filtered results
data = final_data
print(f"Size after sentence existence filter: {len(data)}")

# Sort the labels for consistent output (e.g., by label then by identifier)
semantic_labels_to_export.sort() 

# --- Step 5: Export the semantic labels to a file ---
# Writes the 0-indexed semantic labels and their corresponding file identifiers to the output file,
# separated by a tab.
output_filename = "semantic_labels.txt"
with open(output_filename, 'w') as f:
    for label_tuple in semantic_labels_to_export:
        label, file_identifier = label_tuple
        f.write(f"{label}\t{file_identifier}\n") # Using '0' or '1' as requested
        
print(f"✅ Exported {len(semantic_labels_to_export)} labels to {output_filename}.")

# --- Final Output ---
print(f"\nFinal filtered size: {len(data)}")


# --- Step 6: Verification - Pick 3 random labels and show corresponding sentences ---
print("\n--- Verification: 3 Random Semantic Labels and Sentences ---")

if not semantic_labels_to_export:
    print("No semantic labels to verify.")
else:
    # Load sentences files into memory for quick lookup
    try:
        with open(sentences_file0, 'r') as f0:
            all_sentences0 = f0.readlines()
        with open(sentences_file1, 'r') as f1:
            all_sentences1 = f1.readlines()
    except FileNotFoundError as e:
        print(f"Error during verification: Sentence file not found - {e}")
    else:
        num_samples = min(3, len(semantic_labels_to_export))
        random_samples = random.sample(semantic_labels_to_export, num_samples)

        for i, label_tuple in enumerate(random_samples):
            label, file_identifier = label_tuple
            retrieved_sentence = "N/A"
            source_file_name = ""

            # Get the original sentences from the syntax file that led to this label
            syntax_sentence1, syntax_sentence2 = label_identifier_to_syntax_sentences_map.get(label_tuple, ("N/A", "N/A"))

            if file_identifier == '0':
                source_file_name = os.path.basename(sentences_file0)
                if 0 <= label < len(all_sentences0):
                    retrieved_sentence = all_sentences0[label].strip()
            elif file_identifier == '1':
                source_file_name = os.path.basename(sentences_file1)
                if 0 <= label < len(all_sentences1):
                    retrieved_sentence = all_sentences1[label].strip()
            
            print(f"\nSample {i+1}:")
            print(f"  Label (0-indexed): {label}")
            print(f"  Source File ID: {file_identifier} ({source_file_name})")
            print(f"  Original Sentence 1 (from syntax file): '{syntax_sentence1}'")
            print(f"  Original Sentence 2 (from syntax file): '{syntax_sentence2}'")
            print(f"  Retrieved Sentence (from semantic file): '{retrieved_sentence}'")

print("\n--- Verification Complete ---")


Loaded 3199 unique sentences to filter against.
Size after initial overlap/tag filter: 2369
Size after tag count filter: 2098
Size after sentence existence filter: 1575
✅ Exported 1575 labels to semantic_labels.txt.

Final filtered size: 1575

--- Verification: 3 Random Semantic Labels and Sentences ---

Sample 1:
  Label (0-indexed): 881
  Source File ID: 1 (sentences1.txt)
  Original Sentence 1 (from syntax file): 'A solution could be found eventually'
  Original Sentence 2 (from syntax file): 'The package will be delivered tomorrow'
  Retrieved Sentence (from semantic file): 'A solution could be found eventually'

Sample 2:
  Label (0-indexed): 225
  Source File ID: 1 (sentences1.txt)
  Original Sentence 1 (from syntax file): 'We develop a skill which is impressively useful'
  Original Sentence 2 (from syntax file): 'You built a fence that is wonderfully sturdy'
  Retrieved Sentence (from semantic file): 'We develop a skill which is impressively useful'

Sample 3:
  Label (0-indexed

In [7]:
# 2. Build tag→ID mapping
unique_tags = sorted({tags for _, _, tags, _ in data})
tag_to_id = {tags: idx for idx, tags in enumerate(unique_tags, start=0)}

# 3. Open five files and write out
with open(f"{path0}sentences0.txt", "w") as f1, \
     open(f"{path0}sentences1.txt", "w") as f2, \
     open(f"{path0}group_ids.txt",   "w") as f4:

    for s1, s2, tags, word_overlap in data:
        gid = tag_to_id[tags]
        f1.write(s1 + "\n")
        f2.write(s2 + "\n")
        f4.write(str(gid) + "\n")

In [8]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

all_groups_ids = jnp.array(np.loadtxt('group_ids.txt')).astype(int)[:2000]
unique_groups_indices,counts = jnp.unique(all_groups_ids,return_counts=True)
plt.plot(unique_groups_indices, counts, marker='o')
counts

2025-07-22 16:38:37.594447: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


RuntimeError: Unable to initialize backend 'cuda': FAILED_PRECONDITION: No visible GPU devices. (you may need to uninstall the failing plugin package, or set JAX_PLATFORMS=cpu to skip this backend.)

In [ ]:
[1,2,3,4,5][:15]